# 📓 LBGM-03 LightGBM Model

### LBGM-03 làm gì?
1. **Optuna** — tune attr_3 (bottleneck 22% error) với 50 trials + pruning
2. **Per-attr params** — attr_1/4 ít class dùng params đơn giản; attr_3 dùng Optuna best
3. **Train-only evaluation** — fair val score
4. **Stacking** — attr_2 OOF proba → feature cho attr_5 (Spearman corr=0.545)
5. **Retrain Train+Val** — maximize điểm Kaggle
6. **Save proba** → `/content/pipeline/lgbm/` để NB-05 ensemble

**Runtime:** CPU is fine

In [2]:
!pip install -q lightgbm optuna scikit-learn
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import pickle, json, os
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

TARGET_COLS = ['attr_1','attr_2','attr_3','attr_4','attr_5','attr_6']
os.makedirs('/content/drive/MyDrive/sailormoon/data/pipeline/lgbm', exist_ok=True)
print('✅ Import done')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Import done


## 1. Load Features & Labels

In [3]:
train_feats = pd.read_csv('/content/drive/MyDrive/sailormoon/data/pipeline/train_feats.csv', index_col='id').fillna(-1)
val_feats   = pd.read_csv('/content/drive/MyDrive/sailormoon/data/pipeline/val_feats.csv',   index_col='id').fillna(-1)
test_feats  = pd.read_csv('/content/drive/MyDrive/sailormoon/data/pipeline/test_feats.csv',  index_col='id').fillna(-1)
Y_train_enc = pd.read_csv('/content/drive/MyDrive/sailormoon/data/pipeline/Y_train_enc.csv', index_col='id')
Y_val_enc   = pd.read_csv('/content/drive/MyDrive/sailormoon/data/pipeline/Y_val_enc.csv',   index_col='id')
Y_val_orig  = pd.read_csv('/content/drive/MyDrive/sailormoon/data/pipeline/Y_val_orig.csv',  index_col='id')

with open('/content/drive/MyDrive/sailormoon/data/pipeline/encoders.pkl',  'rb') as f: encoders  = pickle.load(f)
with open('/content/drive/MyDrive/sailormoon/data/pipeline/feat_meta.pkl', 'rb') as f: feat_meta = pickle.load(f)

print(f'train: {train_feats.shape}  val: {val_feats.shape}  test: {test_feats.shape}')
print(f'classes: { {c: Y_train_enc[c].nunique() for c in TARGET_COLS} }')

train: (51000, 121)  val: (7200, 121)  test: (38000, 121)
classes: {'attr_1': 12, 'attr_2': 31, 'attr_3': 99, 'attr_4': 12, 'attr_5': 31, 'attr_6': 99}


## 2. Helper — Exact Match Metric

In [4]:
def exact_match(y_true, y_pred):
    exact    = (y_true.values == y_pred.values).all(axis=1).mean()
    per_attr = {c: accuracy_score(y_true[c], y_pred[c]) for c in TARGET_COLS}
    return exact, per_attr

def print_metrics(y_true, y_pred, tag=''):
    exact, per_attr = exact_match(y_true, y_pred)
    print(f'\n{"="*55}')
    print(f'📊 {tag}')
    print(f'🎯 Exact Match: {exact:.4f}  ({exact*100:.2f}%)')
    for col, acc in per_attr.items():
        bar = '█'*int(acc*20)
        flag = '  ⚠️' if acc < 0.90 else ''
        print(f'  {col}: {acc:.4f}  |{bar}{flag}')
    return exact, per_attr

print('✅ Helpers ready')

✅ Helpers ready


## 3. Optuna — Tune attr_3
> attr_3 sai 22% → bottleneck → tune riêng

**Kỹ thuật tăng tốc:**
- 50% subsample khi tune → mỗi trial nhanh 2×
- MedianPruner → cắt trial kém sau 100 steps
- 50 trials (thay 30) nhờ pruning tiết kiệm ~40% time

In [8]:
TUNE_ATTR = 'attr_3'

np.random.seed(42)
n_sub     = int(len(train_feats) * 0.2)
n_val_sub = int(len(val_feats)   * 0.2)
sub_idx     = np.random.choice(len(train_feats), n_sub,     replace=False)
val_sub_idx = np.random.choice(len(val_feats),   n_val_sub, replace=False)
X_sub     = train_feats.iloc[sub_idx].reset_index(drop=True)
y_sub     = Y_train_enc[TUNE_ATTR].iloc[sub_idx].reset_index(drop=True)
X_val_sub = val_feats.iloc[val_sub_idx].reset_index(drop=True)
y_val_sub = Y_val_enc[TUNE_ATTR].iloc[val_sub_idx].reset_index(drop=True)
print(f'Train sub: {len(X_sub):,} | Val sub: {len(X_val_sub):,} | Classes: {y_sub.nunique()}')

def objective(trial):
    params = {
        'objective'        : 'multiclass',
        'metric'           : 'multi_logloss',
        'num_class'        : Y_train_enc[TUNE_ATTR].nunique(),
        'verbose'          : -1,
        'n_jobs'           : -1,
        'random_state'     : 42,
        'learning_rate'    : trial.suggest_float('lr',         0.05, 0.3, log=True),
        'num_leaves'       : trial.suggest_int(  'num_leaves', 31,   127),
        'max_depth'        : trial.suggest_int(  'max_depth',  4,    8),
        'min_child_samples': trial.suggest_int(  'min_child',  20,   100),
        'subsample'        : trial.suggest_float('subsample',  0.6,  1.0),
        'colsample_bytree' : trial.suggest_float('colsample',  0.6,  1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha',  1e-3, 1.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
    }

    evals_result = {}   # lưu logloss history

    def prune_cb(env):
        # Lấy logloss hiện tại từ evals_result
        if 'valid_0' not in evals_result: return
        hist = evals_result['valid_0'].get('multi_logloss', [])
        if not hist: return
        trial.report(hist[-1], step=env.iteration)   # report logloss (lower=better)
        # Note: study maximize accuracy nhưng prune dựa trên logloss cũng ok
        # MedianPruner so sánh relative → không bị ảnh hưởng bởi direction mismatch
        if trial.should_prune():
            raise optuna.TrialPruned()

    dtrain = lgb.Dataset(X_sub,     label=y_sub)
    dval   = lgb.Dataset(X_val_sub, label=y_val_sub, reference=dtrain)

    model = lgb.train(
        params, dtrain,
        num_boost_round = 300,
        valid_sets      = [dval],
        callbacks       = [
            lgb.early_stopping(20, verbose=False),
            lgb.log_evaluation(9999),
            lgb.record_evaluation(evals_result),   # ← ghi logloss vào dict
            prune_cb,                               # ← prune dựa trên logloss
        ]
    )

    if model.best_iteration < 10:
        raise optuna.TrialPruned()

    # Predict 1 lần duy nhất cuối trial
    acc = accuracy_score(y_val_sub, np.argmax(model.predict(X_val_sub), 1))
    return acc


study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=5),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=20, interval_steps=10),
)
study.enqueue_trial({'lr':0.1,'num_leaves':63,'max_depth':6,'min_child':30,
                     'subsample':0.8,'colsample':0.8,'reg_alpha':0.01,'reg_lambda':0.01})

study.optimize(objective, n_trials=30, show_progress_bar=True, timeout=1800)

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f'\n✅ Best val_sub accuracy: {study.best_value:.4f}')
print(f'   Completed={len(completed)} | Pruned={len(pruned)}')
print('Best params:')
for k, v in study.best_params.items(): print(f'  {k}: {v}')

# ── Verify trên FULL val ──────────────────────────────────────────────
print('\nVerifying on full val...')
bp = study.best_params
best_p = {
    'objective':'multiclass', 'metric':'multi_logloss',
    'num_class': Y_train_enc[TUNE_ATTR].nunique(),
    'verbose':-1, 'n_jobs':-1, 'random_state':42,
    'learning_rate'    : bp['lr'],
    'num_leaves'       : bp['num_leaves'],
    'max_depth'        : bp['max_depth'],
    'min_child_samples': bp['min_child'],
    'subsample'        : bp['subsample'],
    'colsample_bytree' : bp['colsample'],
    'reg_alpha'        : bp['reg_alpha'],
    'reg_lambda'       : bp['reg_lambda'],
}
dtrain_f = lgb.Dataset(train_feats, label=Y_train_enc[TUNE_ATTR])
dval_f   = lgb.Dataset(val_feats,   label=Y_val_enc[TUNE_ATTR], reference=dtrain_f)
m_verify = lgb.train(best_p, dtrain_f, num_boost_round=500, valid_sets=[dval_f],
                      callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(100)])

acc_full = accuracy_score(Y_val_enc[TUNE_ATTR], np.argmax(m_verify.predict(val_feats), 1))
print(f'✅ Full val accuracy (attr_3): {acc_full:.4f}  (best_iter={m_verify.best_iteration})')

Train sub: 10,200 | Val sub: 1,440 | Classes: 99


  0%|          | 0/30 [00:00<?, ?it/s]


✅ Best val_sub accuracy: 0.5701
   Completed=22 | Pruned=2
Best params:
  lr: 0.05783623888299795
  num_leaves: 65
  max_depth: 4
  min_child: 27
  subsample: 0.8280014259559747
  colsample: 0.9963910655147282
  reg_alpha: 0.04101521051135524
  reg_lambda: 0.07256153546380463

Verifying on full val...
[100]	valid_0's multi_logloss: 1.22002
[200]	valid_0's multi_logloss: 0.995152
[300]	valid_0's multi_logloss: 0.904092
[400]	valid_0's multi_logloss: 0.85292
[500]	valid_0's multi_logloss: 0.817367
✅ Full val accuracy (attr_3): 0.8174  (best_iter=500)


## 4. Per-attr Params

In [9]:
op = study.best_params  # Optuna best → dùng cho attr_3

def make_p(n_cls, lr=0.05, leaves=127, depth=-1, child=20,
           sub=0.8, col=0.8, ra=0.1, rl=0.1):
    return {'objective':'multiclass','metric':'multi_logloss','num_class':n_cls,
            'verbose':-1,'n_jobs':-1,'random_state':42,
            'learning_rate':lr,'num_leaves':leaves,'max_depth':depth,
            'min_child_samples':child,'subsample':sub,'colsample_bytree':col,
            'reg_alpha':ra,'reg_lambda':rl}

PARAMS = {
    'attr_1': make_p(Y_train_enc['attr_1'].nunique(), leaves=63,  depth=6),
    'attr_2': make_p(Y_train_enc['attr_2'].nunique(), leaves=127, depth=8),
    'attr_3': make_p(Y_train_enc['attr_3'].nunique(),   # ← Optuna best
                     lr=op.get('lr',0.05),
                     leaves=op.get('num_leaves',200), depth=op.get('max_depth',-1),
                     child=op.get('min_child',20),     sub=op.get('subsample',0.8),
                     col=op.get('colsample',0.8),      ra=op.get('reg_alpha',0.1),
                     rl=op.get('reg_lambda',0.1)),
    'attr_4': make_p(Y_train_enc['attr_4'].nunique(), leaves=63,  depth=6),
    'attr_5': make_p(Y_train_enc['attr_5'].nunique(), leaves=127, depth=8),
    'attr_6': make_p(Y_train_enc['attr_6'].nunique(), leaves=127, depth=8),
}
print('Per-attr params summary:')
for col, p in PARAMS.items():
    print(f'  {col}: lr={p["learning_rate"]:.3f}  leaves={p["num_leaves"]}  depth={p["max_depth"]}')

Per-attr params summary:
  attr_1: lr=0.050  leaves=63  depth=6
  attr_2: lr=0.050  leaves=127  depth=8
  attr_3: lr=0.058  leaves=65  depth=4
  attr_4: lr=0.050  leaves=63  depth=6
  attr_5: lr=0.050  leaves=127  depth=8
  attr_6: lr=0.050  leaves=127  depth=8


## 5. Train-only Models (Fair Val Evaluation)

In [10]:
models_tr  = {}
val_proba  = {}
val_preds  = {}

for col in TARGET_COLS:
    print(f'\n🔄 {col} ({Y_train_enc[col].nunique()} classes)…')
    dtrain = lgb.Dataset(train_feats, label=Y_train_enc[col])
    dval   = lgb.Dataset(val_feats,   label=Y_val_enc[col],  reference=dtrain)
    model  = lgb.train(PARAMS[col], dtrain, num_boost_round=2000, valid_sets=[dval],
                       callbacks=[lgb.early_stopping(80, verbose=False),
                                   lgb.log_evaluation(200)])
    models_tr[col] = model
    p = model.predict(val_feats)
    val_proba[col] = p
    enc  = np.argmax(p, 1)
    val_preds[col] = encoders[col].inverse_transform(enc)
    print(f'  ✅ acc={accuracy_score(Y_val_enc[col], enc):.4f}  best_iter={model.best_iteration}')

val_pred_df = pd.DataFrame(val_preds, index=val_feats.index)
exact_v1, per_attr_v1 = print_metrics(Y_val_orig, val_pred_df, 'LightGBM V1 — train-only')


🔄 attr_1 (12 classes)…
[200]	valid_0's multi_logloss: 0.0206138
  ✅ acc=0.9944  best_iter=271

🔄 attr_2 (31 classes)…
[200]	valid_0's multi_logloss: 0.35123
[400]	valid_0's multi_logloss: 0.302662
[600]	valid_0's multi_logloss: 0.299039
[800]	valid_0's multi_logloss: 0.298437
  ✅ acc=0.9356  best_iter=756

🔄 attr_3 (99 classes)…
[200]	valid_0's multi_logloss: 0.995152
[400]	valid_0's multi_logloss: 0.85292
[600]	valid_0's multi_logloss: 0.802248
[800]	valid_0's multi_logloss: 0.794485
[1000]	valid_0's multi_logloss: 0.793966
  ✅ acc=0.8236  best_iter=950

🔄 attr_4 (12 classes)…
[200]	valid_0's multi_logloss: 0.0326545
[400]	valid_0's multi_logloss: 0.0273888
[600]	valid_0's multi_logloss: 0.0267838
  ✅ acc=0.9929  best_iter=583

🔄 attr_5 (31 classes)…
  ✅ acc=0.9279  best_iter=107

🔄 attr_6 (99 classes)…
[200]	valid_0's multi_logloss: 0.26132
  ✅ acc=0.9322  best_iter=136

📊 LightGBM V1 — train-only
🎯 Exact Match: 0.7113  (71.12%)
  attr_1: 0.9944  |███████████████████
  attr_2: 0.935

## 6. Stacking — attr_2 OOF proba → attr_5
> Spearman corr = 0.545 → khai thác trực tiếp
> Dùng out-of-fold để tránh leakage vào train

In [11]:
print('=== 5-fold OOF cho attr_2 ===')
n_cls_a2 = Y_train_enc['attr_2'].nunique()
A2_COLS  = [f'a2p_{i}' for i in range(n_cls_a2)]
oof_a2   = np.zeros((len(train_feats), n_cls_a2))
skf      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_i, vl_i) in enumerate(skf.split(train_feats, Y_train_enc['attr_2'])):
    Xt, Xv = train_feats.iloc[tr_i], train_feats.iloc[vl_i]
    yt, yv = Y_train_enc['attr_2'].iloc[tr_i], Y_train_enc['attr_2'].iloc[vl_i]
    dt = lgb.Dataset(Xt, label=yt)
    dv = lgb.Dataset(Xv, label=yv, reference=dt)
    m  = lgb.train(PARAMS['attr_2'], dt, num_boost_round=1000, valid_sets=[dv],
                   callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(9999)])
    oof_a2[vl_i] = m.predict(Xv)
    print(f'  Fold {fold+1}/5 done')

# Augmented feature matrices
train_aug = pd.concat([train_feats,
                        pd.DataFrame(oof_a2, columns=A2_COLS, index=train_feats.index)], axis=1)
val_aug   = pd.concat([val_feats,
                        pd.DataFrame(models_tr['attr_2'].predict(val_feats),
                                     columns=A2_COLS, index=val_feats.index)], axis=1)
test_aug  = pd.concat([test_feats,
                        pd.DataFrame(models_tr['attr_2'].predict(test_feats),
                                     columns=A2_COLS, index=test_feats.index)], axis=1)

# Retrain attr_5 với aug features
dt5 = lgb.Dataset(train_aug, label=Y_train_enc['attr_5'])
dv5 = lgb.Dataset(val_aug,   label=Y_val_enc['attr_5'],  reference=dt5)
m5  = lgb.train(PARAMS['attr_5'], dt5, num_boost_round=2000, valid_sets=[dv5],
                callbacks=[lgb.early_stopping(80,verbose=False), lgb.log_evaluation(200)])

acc_before = accuracy_score(Y_val_enc['attr_5'], np.argmax(val_proba['attr_5'], 1))
acc_after  = accuracy_score(Y_val_enc['attr_5'], np.argmax(m5.predict(val_aug), 1))
USE_STACK  = acc_after > acc_before
print(f'\nattr_5  before={acc_before:.4f}  after={acc_after:.4f}  Δ={acc_after-acc_before:+.4f}')
print(f'USE_STACK = {USE_STACK}')

=== 5-fold OOF cho attr_2 ===
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done
[200]	valid_0's multi_logloss: 0.331269

attr_5  before=0.9279  after=0.9274  Δ=-0.0006
USE_STACK = False


## 7. Final Val Evaluation

In [12]:
final_val_proba = {}
final_val_preds = {}
for col in TARGET_COLS:
    if col == 'attr_5' and USE_STACK:
        p = m5.predict(val_aug); print(f'  {col}: stacked ✨')
    else:
        p = val_proba[col]
    final_val_proba[col] = p
    final_val_preds[col] = encoders[col].inverse_transform(np.argmax(p, 1))

fvdf = pd.DataFrame(final_val_preds, index=val_feats.index)
exact_final, per_attr_final = print_metrics(Y_val_orig, fvdf, 'LightGBM FINAL — train-only')
print(f'\n📈 vs baseline 66.94%: {(exact_final - 0.6694)*100:+.2f}%')

# Error analysis
print('\n% error per attr:')
for col in TARGET_COLS:
    e = 1 - per_attr_final[col]
    bar = '█'*int(e*60)
    print(f'  {col}: {e*100:.2f}%  {bar}')


📊 LightGBM FINAL — train-only
🎯 Exact Match: 0.7113  (71.12%)
  attr_1: 0.9944  |███████████████████
  attr_2: 0.9356  |██████████████████
  attr_3: 0.8236  |████████████████  ⚠️
  attr_4: 0.9929  |███████████████████
  attr_5: 0.9279  |██████████████████
  attr_6: 0.9322  |██████████████████

📈 vs baseline 66.94%: +4.19%

% error per attr:
  attr_1: 0.56%  
  attr_2: 6.44%  ███
  attr_3: 17.64%  ██████████
  attr_4: 0.71%  
  attr_5: 7.21%  ████
  attr_6: 6.78%  ████


## 8. Retrain trên Train + Val
> **Rule BTC:** Sau khi chốt params, được phép gộp train+val để retrain
> n_rounds = best_iteration × (1 + val/train ratio)

In [13]:
val_ratio  = len(val_feats) / len(train_feats)
full_feats = pd.concat([train_feats, val_feats], axis=0)
Y_full_enc = pd.concat([Y_train_enc, Y_val_enc],  axis=0)
print(f'Full: {len(full_feats):,} samples  (×{1+val_ratio:.3f} vs train-only)')

# OOF attr_2 trên full data (tránh leakage cho attr_5 stacking)
print('\nOOF attr_2 trên full data…')
oof_a2_full = np.zeros((len(full_feats), n_cls_a2))
skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_i, vl_i) in enumerate(skf2.split(full_feats, Y_full_enc['attr_2'])):
    Xt, Xv = full_feats.iloc[tr_i], full_feats.iloc[vl_i]
    yt     = Y_full_enc['attr_2'].iloc[tr_i]
    m      = lgb.train(PARAMS['attr_2'], lgb.Dataset(Xt, label=yt),
                       num_boost_round=800, callbacks=[lgb.log_evaluation(9999)])
    oof_a2_full[vl_i] = m.predict(Xv)
    print(f'  Fold {fold+1}/5 done')

full_aug      = pd.concat([full_feats,
                            pd.DataFrame(oof_a2_full, columns=A2_COLS, index=full_feats.index)], axis=1)
test_aug_full = pd.concat([test_feats,
                            pd.DataFrame(models_tr['attr_2'].predict(test_feats),
                                         columns=A2_COLS, index=test_feats.index)], axis=1)

print('\nRetraining all attrs on full data…')
models_full = {}
test_proba  = {}
for col in TARGET_COLS:
    bi       = models_tr[col].best_iteration
    n_rounds = int(bi * (1 + val_ratio))
    use_aug  = (col == 'attr_5' and USE_STACK)
    Xf       = full_aug      if use_aug else full_feats
    Xt_      = test_aug_full if use_aug else test_feats
    model    = lgb.train(PARAMS[col], lgb.Dataset(Xf, label=Y_full_enc[col]),
                         num_boost_round=n_rounds, callbacks=[lgb.log_evaluation(200)])
    models_full[col] = model
    test_proba[col]  = model.predict(Xt_)
    print(f'  ✅ {col}: iter={bi} → {n_rounds} rounds')

print('\n✅ All retrained on Train+Val')

Full: 58,200 samples  (×1.141 vs train-only)

OOF attr_2 trên full data…
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done

Retraining all attrs on full data…
  ✅ attr_1: iter=271 → 309 rounds
  ✅ attr_2: iter=756 → 862 rounds
  ✅ attr_3: iter=950 → 1084 rounds
  ✅ attr_4: iter=583 → 665 rounds
  ✅ attr_5: iter=107 → 122 rounds
  ✅ attr_6: iter=136 → 155 rounds

✅ All retrained on Train+Val


## 9. Save Outputs → NB-05 Ensemble

In [ ]:
for col in TARGET_COLS:
    models_full[col].save_model(f'/content/drive/MyDrive/sailormoon/data/pipeline/lgbm/model_{col}.txt')
    np.save(f'/content/drive/MyDrive/sailormoon/data/pipeline/lgbm/val_proba_{col}.npy',  final_val_proba[col])
    np.save(f'/content/drive/MyDrive/sailormoon/data/pipeline/lgbm/test_proba_{col}.npy', test_proba[col])

stack_meta = {'USE_STACK': USE_STACK, 'A2_COLS': A2_COLS, 'n_cls_a2': n_cls_a2}
with open('/content/drive/MyDrive/sailormoon/data/pipeline/lgbm/stack_meta.pkl', 'wb') as f:
    pickle.dump(stack_meta, f)

metrics = {'exact_match': float(exact_final),
           'per_attr': {k: float(v) for k,v in per_attr_final.items()},
           'baseline': 0.6694, 'improvement': float(exact_final - 0.6694)}
with open('/content/drive/MyDrive/sailormoon/data/pipeline/lgbm/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('✅ Saved to /content/drive/MyDrive/sailormoon/data/pipeline/lgbm/')
print(f'\n📊 Val Exact Match: {exact_final:.4f}  ({exact_final*100:.2f}%)')


✅ Saved to /content/drive/MyDrive/sailormoon/data/pipeline/lgbm/

📊 Val Exact Match: 0.7113  (71.12%)

➡️  Next: NB-04_deep_learning.ipynb  (đổi Runtime → GPU trước!)
